In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
from shutil import rmtree
import numpy as np
import requests
import time
import json


import hydra
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf
import pyrootutils
import polars as pl
from tqdm import tqdm
import concurrent.futures

from datasets import (
    ClassLabel,
    Dataset,
    DatasetDict,
    Features,
    Image,
    Sequence,
    Value,
    concatenate_datasets,
    load_dataset,
    load_from_disk
)

from planktonzilla.utils.logger import get_pylogger
from planktonzilla.dataset_import.dataset_importer import (
    DatasetImporter,
    is_dir_empty,
    is_valid_image_file,
)

In [1]:
import open_clip

clip_model, _, _ = open_clip.create_model_and_transforms('hf-hub:project-oceania/CLIP-EVA02-L-14.planktonzilla-pt')

/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/

RuntimeError: Failed initial config/weights load from HF Hub project-oceania/CLIP-EVA02-L-14.planktonzilla-pt: Failed to download file (open_clip_config.json) for project-oceania/CLIP-EVA02-L-14.planktonzilla-pt. Last error: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.

In [1]:
from datasets import ClassLabel, Dataset, Sequence, concatenate_datasets, load_dataset, Features, Image, Value, DatasetDict, load_from_disk
from joblib import Parallel, delayed
from tqdm import tqdm

from datasets import concatenate_datasets
import numpy as np
import os

import os
import io
import tarfile
from io import BytesIO
from PIL import Image
from tqdm import tqdm 


def complete_taxonomy(example, taxonomy_levels):
    """
    Rellena niveles taxonómicos faltantes usando:
    unknown_{last_known_taxo}_{current_level}
    """
    last_known = None
    
    for level in taxonomy_levels:
        value = example.get(level)

        # Consideramos vacío si es None, string vacío o NaN-like
        if value is None or str(value).strip() == "" or str(value).lower() == "nan":
            if last_known is not None:
                example[level] = f"unknown_{last_known.lower()}_{level.lower()}"
        else:
            # Solo actualizamos last_known si NO es un unknown generado
            if not str(value).startswith("unknown_"):
                last_known = str(value)

    return example



def export_to_tar_shards(dataset_dict, output_dir="data", shard_size=1_000):
    """
    Exporta un DatasetDict a shards .tar para entrenamiento tipo CLIP/WebDataset,
    asegurando que todas las imágenes se guarden en formato RGB.
    """
    os.makedirs(output_dir, exist_ok=True)

    for split_name, dataset in dataset_dict.items():
        split_dir = os.path.join(output_dir, split_name)
        os.makedirs(split_dir, exist_ok=True)

        total_samples = len(dataset)
        n_shards = (total_samples + shard_size - 1) // shard_size
        
        taxo_classes = dataset.features["label"].names

        for shard_idx in range(n_shards):
            start = shard_idx * shard_size
            end = min((shard_idx + 1) * shard_size, total_samples)

            shard_path = os.path.join(split_dir, f"shard_{shard_idx:05d}.tar")
            shard_indices = range(start, end)
            
            with tarfile.open(shard_path, "w") as tar:
                # Iterar sobre los índices absolutos del dataset
                for i_abs in tqdm(shard_indices, desc=f"{split_name} shard {shard_idx}"):

                    example = dataset[i_abs]

                    i = i_abs - start # Índice relativo dentro del shard

                    # --- 1. Imagen (key: image_{i}.jpg) ---
                    img = example["image"]
                    if not isinstance(img, Image.Image):
                        raise ValueError(f"El campo 'image' en el índice {i_abs} no es un objeto PIL.Image")

                    # *** CAMBIO CLAVE: Convertir la imagen a RGB antes de guardar ***
                    # Esto maneja imágenes en escala de grises (L) o con paleta (P), o RGBA
                    img_rgb = img.convert('RGB')
                    
                    # Guardar la imagen en un buffer de bytes
                    img_bytes = io.BytesIO()
                    img_rgb.save(img_bytes, format="PNG") 
                    img_bytes.seek(0)

                    # Crear el TarInfo y añadir el archivo de imagen
                    img_info = tarfile.TarInfo(name=f"image_{i}.png")
                    img_info.size = len(img_bytes.getbuffer())
                    tar.addfile(img_info, img_bytes)

                    # --- 2. Etiqueta/Texto (key: text_{i}.txt) ---
                    label_str = str(taxo_classes[example["label"]])
                    label_bytes = io.BytesIO(label_str.encode("utf-8"))
                    
                    # Crear el TarInfo y añadir el archivo de texto
                    label_info = tarfile.TarInfo(name=f"image_{i}.txt")
                    label_info.size = len(label_bytes.getbuffer())
                    tar.addfile(label_info, label_bytes)


dataset = load_from_disk("/lustre/fsn1/projects/rech/tec/uod68bo/data/planktonzilla_only_plankton2")
del dataset["test"]
del dataset["train"]

taxonomy_levels = [
    "Kingdom",
    "Phylum",
    "Class",
    "Order",
    "Family",
    "Genus",
    "Species",
]

dataset = dataset.map(
    lambda x: complete_taxonomy(x, taxonomy_levels),
    num_proc=32
)

/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map (num_proc=32): 100%|██████████| 590526/590526 [01:23<00:00, 7081.68 examples/s]  


In [9]:
taxonomy_levels = [
    "Kingdom",
    "Phylum",
    "Class",
    "Order",
    "Family",
    "Genus",
    "Species",
]

def expand_label_name(label_str, taxonomy_levels):
    tokens = label_str.lower().split()
    expanded = []
    last_known = None

    for i, level in enumerate(taxonomy_levels):
        if i < len(tokens):
            expanded.append(tokens[i])
            last_known = tokens[i]
        else:
            expanded.append(f"unknown_{last_known}_{level.lower()}")

    return " ".join(expanded)

old_names = dataset["validation"].features["label"].names

new_names = [
    expand_label_name(name, taxonomy_levels)
    for name in old_names
]

In [10]:
new_names

['animalia annelida unknown_annelida_class unknown_annelida_order unknown_annelida_family unknown_annelida_genus unknown_annelida_species',
 'animalia annelida polychaeta unknown_polychaeta_order unknown_polychaeta_family unknown_polychaeta_genus unknown_polychaeta_species',
 'animalia annelida polychaeta phyllodocida unknown_phyllodocida_family unknown_phyllodocida_genus unknown_phyllodocida_species',
 'animalia annelida polychaeta phyllodocida tomopteridae unknown_tomopteridae_genus unknown_tomopteridae_species',
 'animalia annelida polychaeta terebellida acrocirridae swima unknown_swima_species',
 'animalia annelida polychaeta terebellida flabelligeridae poeobius unknown_poeobius_species',
 'animalia arthropoda unknown_arthropoda_class unknown_arthropoda_order unknown_arthropoda_family unknown_arthropoda_genus unknown_arthropoda_species',
 'animalia arthropoda arachnida trombidiformes unknown_trombidiformes_family unknown_trombidiformes_genus unknown_trombidiformes_species',
 'anima

In [5]:
dataset["validation"][0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=L size=49x291>,
 'label': 132,
 'dataset': 'flowcamnet'}

In [7]:
dataset["validation"].features["label"].names

['animalia annelida',
 'animalia annelida polychaeta',
 'animalia annelida polychaeta phyllodocida',
 'animalia annelida polychaeta phyllodocida tomopteridae',
 'animalia annelida polychaeta terebellida acrocirridae swima',
 'animalia annelida polychaeta terebellida flabelligeridae poeobius',
 'animalia arthropoda',
 'animalia arthropoda arachnida trombidiformes',
 'animalia arthropoda arachnida trombidiformes halacaridae',
 'animalia arthropoda branchiopoda anomopoda bosminidae bosmina',
 'animalia arthropoda branchiopoda anomopoda daphniidae ceriodaphnia',
 'animalia arthropoda branchiopoda anomopoda daphniidae daphnia',
 'animalia arthropoda branchiopoda ctenopoda sididae diaphanosoma',
 'animalia arthropoda branchiopoda ctenopoda sididae penilia',
 'animalia arthropoda branchiopoda ctenopoda sididae penilia avirostris',
 'animalia arthropoda branchiopoda haplopoda leptodoridae leptodora',
 'animalia arthropoda branchiopoda onychopoda podonidae evadne',
 'animalia arthropoda branchi

In [1]:
from datasets import load_dataset

/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("project-oceania/planktonzilla_only_plankton")

Generating test split: 100%|██████████| 590532/590532 [00:12<00:00, 46850.53 examples/s] 


In [ ]:
root = pyrootutils.setup_root(
    search_from=".",  
    indicator=[".git", "pyproject.toml"],
    pythonpath=True,
    dotenv=True,
)

logger = get_pylogger(__name__)

In [ ]:
def custom_cleanup(self):
    if self.cleanup_after_processing:
        logger.info("Removing downloaded and intermediate files.")
        if self.download_manager:
            self.download_manager.delete_extracted_files()

        if self.raw_dir and self.raw_dir.exists():
            rmtree(self.raw_dir, ignore_errors=True)

    else:
        logger.info("Keeping downloaded and intermediate files, set cleanup_after_processing=True to change this.")

In [1]:
from datasets import (
    ClassLabel,
    Dataset,
    DatasetDict,
    Features,
    Image,
    Sequence,
    Value,
    concatenate_datasets,
    load_dataset,
    load_from_disk
)
dataset = load_dataset("project-oceania/planktonzilla_only_plankton")

/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split:  10%|▉         | 283915/2952669 [00:58<12:54, 3445.71 examples/s] 

: 

In [ ]:
def custom_import_dataset(self):
    """
    Versión modificada de import_dataset que NO carga HF ni sube al Hub.
    """
    self._download_and_extract()

    if not self.force_imagefolder_preparation and not is_dir_empty(self.imagefolder_dir):
        logger.info(
            f"ImageFolder dir {self.imagefolder_dir} is not empty, using its content as dataset. Set force_imagefolder_preparation=True to change this."
        )
    else:
        logger.info(f"Preparing dataset as imagefolder in {self.imagefolder_dir}")
        self._prepare_imagefolder()

    if self.check_image_file_integrity:
        for class_dir in tqdm(
            os.listdir(self.imagefolder_dir),
            desc="Validating classes.",
            disable=not self.show_progress,
            leave=False,
        ):
            for file in tqdm(
                os.listdir(self.imagefolder_dir / class_dir),
                disable=not self.show_progress,
                leave=False,
            ):
                if not is_valid_image_file(self.imagefolder_dir / class_dir / file):
                    logger.warning(f"Invalid file {file} in class {class_dir} detected. Removing it from dataset.")
                    os.remove(self.imagefolder_dir / class_dir / file)
    
    self.cleanup()


In [ ]:
DatasetImporter.import_dataset = custom_import_dataset
DatasetImporter.cleanup = custom_cleanup

In [ ]:
# Note: the Zoolake and SYKE Zooscan 2024 URL has an anti-bot protection, so it cannot be downloaded using Python libraries. 
# Therefore, you must manually download the .zip file, add it to the repository, and provide the file path.
# zoolake url: https://opendata.eawag.ch/dataset/52b6ba86-5ecb-448c-8c01-eec7cb209dc7/resource/1cc785fa-36c2-447d-bb11-92ce1d1f3f2d/download/data.zip
# SYKE Zooscan 2024 url: https://etsin.fairdata.fi/dataset/6fa42787-9772-41a5-a6fc-0dde489ed908/data

path_zip_zoolake = "/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/data/data.zip"
path_zip_syze_zooscan2024 = "/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/data/SYKE-plankton_ZooScan_2024.zip"

In [ ]:
datasets_configs = {
     "planktoscope": {
         "overrides": [
             "dataset_import=planktoscope",
             "dataset_import.cleanup_after_processing=True",
             "dataset_import.data_dir=/lustre/fsn1/projects/rech/tec/uod68bo/data/"
         ],
     },
}


In [ ]:
with hydra.initialize(version_base="1.3", config_path="../configs"):
    
    for dataset_name, ds_cfg in datasets_configs.items():
        print(f"\n=== Dataset: {dataset_name} ===")
        cfg = hydra.compose(
            config_name="import_dataset",  
            overrides=ds_cfg["overrides"]
        )

        dataset_importer = hydra.utils.instantiate(cfg.dataset_import)
        dataset_importer.import_dataset() # now this only generates the imagefolder

        #dataset = load_dataset("imagefolder", data_dir=dataset_importer.imagefolder_dir)
        #print("redefine")
        #dataset = r.redefine(hf_dataset=dataset,dataset_name=dataset_name)

        #dataset.save_to_disk(dataset_name)

        #datasets_list.append(final_dataset)

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/notebooks/oceans_cpis_validated.csv")

In [ ]:
dfdf[]

In [ ]:
for label in df["label"].unique():

    largo = df[df["label"]==label][['decimal_latitude', 'decimal_longitude', 'minimum_depth_in_meters',
       'maximum_depth_in_meters']].drop_duplicates().shape[0]
    if largo>1:
        print(label,largo)

In [ ]:
df[['decimal_latitude', 'decimal_longitude', 'minimum_depth_in_meters',
       'maximum_depth_in_meters', 'label']].drop_duplicates()

In [ ]:
df[['decimal_latitude', 'decimal_longitude', 'minimum_depth_in_meters',
       'maximum_depth_in_meters']].drop_duplicates()

In [ ]:
df[df["label"]=="26.Chaetognatha"][['decimal_latitude', 'decimal_longitude', 'minimum_depth_in_meters',
       'maximum_depth_in_meters']].drop_duplicates().shape[0]

In [ ]:
df.iloc[0]["label"]

# ./CPICS_Validated/20141001-07/LClass_26.Chaetognatha/20141001_131452.622.0.png

In [ ]:
merged_dataset = concatenate_datasets(datasets_list)

## Push to HF

In [ ]:
import sys
import os

root_path = os.path.abspath("..")
sys.path.append(root_path)

from planktonzilla.dataset_import.dataset_importer import *

In [ ]:
from datasets import load_from_disk, concatenate_datasets, DatasetDict
from pathlib import Path

BASE_DIR = Path("/lustre/fsn1/projects/rech/tec/uod68bo/data")

# Detecta automáticamente todos los datasets *_hf
hf_paths = sorted(p for p in BASE_DIR.glob("*_hf") if p.is_dir())

datasets = [load_from_disk(p) for p in hf_paths]
datasets = concatenate_datasets(datasets)

In [ ]:
hf_dataset = DatasetDict({
    "train": datasets,
})

In [ ]:
def report_dataset_content_custom(huggingface_dataset: Dataset | DatasetDict) -> str:
    def report_split(dataset: Dataset, split_name: str) -> str:
        class_names, class_counts = np.unique(dataset["original_label"], return_counts=True)

        content = []
        for class_idx, class_name in enumerate(class_names):
            content += [f"{class_idx}: {class_name}"]

        plt.simple_bar(content, class_counts.astype(int), title=f"Label histogram for {split_name} split ", width=83)
        plt.show()

        return strip_ansi_codes(plt.build())

    if isinstance(huggingface_dataset, DatasetDict):
        split_reports = []
        split_reports = [
            f"**Samples per class for split `{split}`**\n ```{report_split(huggingface_dataset[split], split)}```\n"
            for split in huggingface_dataset
        ]
        return "\n".join(split_reports)
    else:
        return report_split(huggingface_dataset) + "\n"

In [ ]:
import planktonzilla.dataset_import.dataset_importer as importer_module

# Reemplazamos la función original por la personalizada en el contexto del módulo
importer_module.report_dataset_content = report_dataset_content_custom

In [ ]:
@dataclass
class PlanktonzillaDatasetImporter(DatasetImporter):
    def import_dataset(self, hf_dataset):
        self.hf_dataset = hf_dataset
        self._push_to_hub()

In [ ]:
from huggingface_hub.utils import get_token
hf_token = get_token()

In [ ]:
hf_dataset = DatasetDict({
    "train": datasets,
})

In [ ]:
importer = PlanktonzillaDatasetImporter(
    data_dir="",
    hf_token =hf_token,
    hf_private=False,
    push_to_hub=True,
    hf_dataset_name="ecotaxa_planktonzilla",
    hf_org_name="project-oceania",
    human_readable_name="Plankton Dataset",
    description = "A dataset composed of all publicly available, labeled EcoTaxa datasets",
    source_url="https://huggingface.co/datasets/project-oceania/ecotaxa_planktonzilla",
    license="cc-by-4.0",
    citation_bibtex="""@misc{ecotaxa_planktonzilla,
        author       = {Inria Chile},
        title        = {Ecotaxa planktonzilla dataset},
        month        = jan,
        year         = 2026,
        version      = {1.0.0},
        doi          = {TBD},
        url          = {https://huggingface.co/datasets/project-oceania/ecotaxa_planktonzilla},
    }""",
)

importer.import_dataset(hf_dataset)

In [1]:
import sys
import os

root_path = os.path.abspath("..")
sys.path.append(root_path)

from planktonzilla.dataset_import.dataset_importer import *


from datasets import load_from_disk, concatenate_datasets, DatasetDict, Dataset, Value
from pathlib import Path


def to_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, str):
        return x.lower() == "true"
    return False


def normalize_example(example):
    example["plankton"] = to_bool(example["plankton"])
    example["living"]   = to_bool(example["living"])
    return example



BASE_DIR = Path("/lustre/fsn1/projects/rech/tec/uod68bo/data")

# Detecta automáticamente todos los datasets *_hf
hf_paths = sorted(p for p in BASE_DIR.glob("*_hf") if p.is_dir())

datasets = []

for p in hf_paths:
    print(p)
    d = load_from_disk(p)
    datasets.append(d)

/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/lustre/fsn1/projects/rech/tec/uod68bo/data/flowcamnet_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/isiisnet_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/jedioceans_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/lensless_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/medplanktonset_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/sykezooscan2024_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/uvp6net_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/whoi_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/zoocamnet_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/zoolake_hf
/lustre/fsn1/projects/rech/tec/uod68bo/data/zooscan_hf


: 

In [ ]:
class_names, class_counts = np.unique(dataset["proposed_label"], return_counts=True)

In [ ]:
import sys
import os

root_path = os.path.abspath("..")
sys.path.append(root_path)

from planktonzilla.dataset_import.dataset_importer import *


from datasets import load_from_disk, concatenate_datasets, DatasetDict, Dataset, Value
from pathlib import Path


def to_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, str):
        return x.lower() == "true"
    return False


def normalize_example(example):
    example["plankton"] = to_bool(example["plankton"])
    example["living"]   = to_bool(example["living"])
    return example



BASE_DIR = Path("/lustre/fsn1/projects/rech/tec/uod68bo/data")

# Detecta automáticamente todos los datasets *_hf
hf_paths = sorted(p for p in BASE_DIR.glob("*_hf") if p.is_dir())

datasets = []

for p in hf_paths[:3]:
    d = load_from_disk(p)

    d = d.map(
        normalize_example,
        num_proc=16,
        desc="Normalizing bool columns",
    )

    datasets.append(d)

datasets = concatenate_datasets(datasets)


def report_dataset_content_custom(huggingface_dataset: Dataset | DatasetDict) -> str:
    def report_split(dataset: Dataset, split_name: str) -> str:
        class_names, class_counts = np.unique(dataset["proposed_label"], return_counts=True)

        content = []
        for class_idx, class_name in enumerate(class_names):
            content += [f"{class_idx}: {class_name}"]

        plt.simple_bar(content, class_counts.astype(int), title=f"Label histogram for {split_name} split ", width=83)
        plt.show()

        return strip_ansi_codes(plt.build())

    if isinstance(huggingface_dataset, DatasetDict):
        split_reports = []
        split_reports = [
            f"**Samples per class for split `{split}`**\n ```{report_split(huggingface_dataset[split], split)}```\n"
            for split in huggingface_dataset
        ]
        return "\n".join(split_reports)
    else:
        return report_split(huggingface_dataset) + "\n"


import planktonzilla.dataset_import.dataset_importer as importer_module

# Reemplazamos la función original por la personalizada en el contexto del módulo
importer_module.report_dataset_content = report_dataset_content_custom



@dataclass
class PlanktonzillaDatasetImporter(DatasetImporter):
    def import_dataset(self, hf_dataset):
        self.hf_dataset = hf_dataset
        self._push_to_hub()


from huggingface_hub.utils import get_token
hf_token = get_token()


hf_dataset = DatasetDict({
    "train": datasets,
})


importer = PlanktonzillaDatasetImporter(
    data_dir="",
    hf_token =hf_token,
    hf_private=False,
    push_to_hub=True,
    hf_dataset_name="planktonzilla_full",
    hf_org_name="project-oceania",
    human_readable_name="Plankton Dataset",
    description = "A dataset composed of all publicly available, labeled plankton datasets",
    source_url="https://huggingface.co/datasets/project-oceania/planktonzilla_full",
    license="cc-by-4.0",
    citation_bibtex="""@misc{planktonzilla_full,
        author       = {Inria Chile},
        title        = {Planktonzilla dataset},
        month        = jan,
        year         = 2026,
        version      = {1.0.0},
        doi          = {TBD},
        url          = {https://huggingface.co/datasets/project-oceania/planktonzilla_full},
    }""",
)



def main():
    importer.import_dataset(hf_dataset)
    print("DONE")
if __name__ == "__main__":
    main()
